In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("PySparkPractice").getOrCreate()

employee_data = [
    (101, "Sravan", "Data Engineer", "IT", 75000, "Hyderabad", 28, "2021-05-10", "Male"),
    (102, "Ravi", "Software Engineer", "IT", 68000, "Bangalore", 30, "2020-03-15", "Male"),
    (103, "Priya", "Data Analyst", "Analytics", 62000, "Chennai", 26, "2022-01-12", "Female"),
    (104, "Kiran", "Manager", "HR", 90000, "Mumbai", 35, "2018-07-19", "Male"),
    (105, "Anjali", "HR Executive", "HR", 45000, "Pune", 24, "2023-02-20", "Female"),
    (106, "Vikram", "Data Scientist", "Analytics", 98000, "Delhi", 32, "2019-11-25", "Male"),
    (107, "Sneha", "Developer", "IT", 71000, "Hyderabad", 27, "2021-08-17", "Female"),
    (108, "Rahul", "Tester", "QA", 55000, "Chennai", 29, "2020-06-10", "Male"),
    (109, "Meena", "QA Lead", "QA", 83000, "Bangalore", 33, "2017-09-14", "Female"),
    (110, "Arjun", "Support Engineer", "Support", 50000, "Pune", 31, "2022-04-11", "Male")
]

columns = ["emp_id","name","designation","department","salary","city","age","joining_date","gender"]

emp_df = spark.createDataFrame(employee_data, columns)

df = emp_df

In [0]:
# Count Male and Female Employees
df.groupBy("gender") \
  .count() \
  .show()

+------+-----+
|gender|count|
+------+-----+
|  Male|    6|
|Female|    4|
+------+-----+



In [0]:
# Average Age Department-wise
df.groupBy("department") \
  .agg(
      avg("age").alias("average_age")
  ) \
  .show()

+----------+------------------+
|department|       average_age|
+----------+------------------+
|        IT|28.333333333333332|
| Analytics|              29.0|
|        HR|              29.5|
|        QA|              31.0|
|   Support|              31.0|
+----------+------------------+



In [0]:
# Hiring Trends by Year
df.withColumn(
    "joining_year",
    year(to_date(col("joining_date")))
).groupBy(
    "joining_year"
).count().show()

+------------+-----+
|joining_year|count|
+------------+-----+
|        2021|    2|
|        2020|    2|
|        2022|    2|
|        2018|    1|
|        2023|    1|
|        2019|    1|
|        2017|    1|
+------------+-----+



In [0]:
# City-wise Employee Distribution
df.groupBy("city") \
  .count() \
  .show()

+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    2|
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|     Pune|    2|
|    Delhi|    1|
+---------+-----+



In [0]:
# Highest Paid Employees
max_salary = df.agg(
    max("salary")
).collect()[0][0]

df.filter(
    col("salary") == max_salary
).show()

+------+------+--------------+----------+------+-----+---+------------+------+
|emp_id|  name|   designation|department|salary| city|age|joining_date|gender|
+------+------+--------------+----------+------+-----+---+------------+------+
|   106|Vikram|Data Scientist| Analytics| 98000|Delhi| 32|  2019-11-25|  Male|
+------+------+--------------+----------+------+-----+---+------------+------+



In [0]:
# Lowest Paid Employees
min_salary = df.agg(
    min("salary")
).collect()[0][0]

df.filter(
    col("salary") == min_salary
).show()

+------+------+------------+----------+------+----+---+------------+------+
|emp_id|  name| designation|department|salary|city|age|joining_date|gender|
+------+------+------------+----------+------+----+---+------------+------+
|   105|Anjali|HR Executive|        HR| 45000|Pune| 24|  2023-02-20|Female|
+------+------+------------+----------+------+----+---+------------+------+



In [0]:
# Department-wise Employee Count
df.groupBy("department") \
  .count() \
  .show()

+----------+-----+
|department|count|
+----------+-----+
|        IT|    3|
| Analytics|    2|
|        HR|    2|
|        QA|    2|
|   Support|    1|
+----------+-----+



In [0]:
# Gender-wise Average Salary
df.groupBy("gender") \
  .agg(
      avg("salary").alias("average_salary")
  ) \
  .show()

+------+-----------------+
|gender|   average_salary|
+------+-----------------+
|  Male|72666.66666666667|
|Female|          65250.0|
+------+-----------------+



In [0]:
# Employees Above Average Salary
avg_salary = df.agg(
    avg("salary")
).collect()[0][0]

df.filter(
    col("salary") > avg_salary
).show()

+------+------+--------------+----------+------+---------+---+------------+------+
|emp_id|  name|   designation|department|salary|     city|age|joining_date|gender|
+------+------+--------------+----------+------+---------+---+------------+------+
|   101|Sravan| Data Engineer|        IT| 75000|Hyderabad| 28|  2021-05-10|  Male|
|   104| Kiran|       Manager|        HR| 90000|   Mumbai| 35|  2018-07-19|  Male|
|   106|Vikram|Data Scientist| Analytics| 98000|    Delhi| 32|  2019-11-25|  Male|
|   107| Sneha|     Developer|        IT| 71000|Hyderabad| 27|  2021-08-17|Female|
|   109| Meena|       QA Lead|        QA| 83000|Bangalore| 33|  2017-09-14|Female|
+------+------+--------------+----------+------+---------+---+------------+------+



In [0]:
# HR Summary Dashboard Report
df.groupBy("department") \
  .agg(
      count("*").alias("employee_count"),
      avg("salary").alias("avg_salary"),
      avg("age").alias("avg_age")
  ) \
  .show()

+----------+--------------+-----------------+------------------+
|department|employee_count|       avg_salary|           avg_age|
+----------+--------------+-----------------+------------------+
|        IT|             3|71333.33333333333|28.333333333333332|
| Analytics|             2|          80000.0|              29.0|
|        HR|             2|          67500.0|              29.5|
|        QA|             2|          69000.0|              31.0|
|   Support|             1|          50000.0|              31.0|
+----------+--------------+-----------------+------------------+

